In [1]:
"""
stage_5.py -- Stage 5 (iteration 2): 5c2-raw hazard model, manifest-driven.

Frozen model contract (5c2-raw): grammar (bsince + ewm{2,6,18} per class, age, tod)
+ causal z-scored values @ {t-1, t-2, t-3}: signed & magnitude of every stream's
source column, leg amplitude, raw body signed & magnitude. Sign convention
confirming-positive (value * leg_dir).

Manifest-driven (iteration 2):
  - Fork axes (frame, session window, stream set) are READ from the Stage-0
    manifest, never redeclared here. Dropping a stream in Stage 0 (e.g. no-TICK)
    propagates automatically: grammar classes AND z-value channels both track the
    manifest stream set.
  - tod is derived from clock time (session_start), resolution-independent.
  - The manifest is baked into the model bundle so the booster carries its own
    contract; the worker asserts against it and prints it at startup.

Naming (iteration 2):
  SOURCE_PATH / src : the "source" oscillator file (HA OHLC + JMA + TICK + derivs),
                      Stage-0's input; rawer than bars/events. src is the primary
                      data frame, augmented in-place with raw body columns.
  RAW_FILE / raw1   : pure NQ raw OHLC (transient; merged into src, then unused).
"""

import json
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from scipy.signal import lfilter
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss, roc_auc_score

from common import Featurizer, load_manifest, _expanding_z, _welford_check

In [2]:
# ---------------------------------------------------------------- CONFIG (per-run, explicit)
FRAME = 3
STAGE0_TAG = 'nq-TICK-9-12am'

MANIFEST_PATH = f"stage-0/manifest_{STAGE0_TAG}_{FRAME}s.json"
BARS_PATH = f"stage-0/bars_{STAGE0_TAG}_{FRAME}s.pqt"
EVENTS_PATH = f"stage-0/events_{STAGE0_TAG}_{FRAME}s.pqt"

ITER_DIR = "."                                   # iteration-2 root (encapsulated)
OUT_DIR = "stage-5"

VALID_FROM = "2025-07-01"
TRAIN_END = "2025-12-31"
TEST_FROM = "2026-01-01"

# frozen 5c2 architecture constants (NOT fork axes -- stay in code)
TAUS = (2.0, 6.0, 18.0)
VALUE_LAGS = (1, 2)                               # t-2, t-3 (t-1 shift is implicit)
ZWARM = 20
TOD_BIN_MIN = 30
BODY_OPEN_COL = "rawOpen"
BODY_CLOSE_COL = "rawLast"
BODY_TAG = "raw"

LGBM_PARAMS = dict(
    objective="binary", metric="binary_logloss", learning_rate=0.05,
    num_leaves=127, min_data_in_leaf=1000, feature_fraction=0.9,
    bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
    num_threads=16, verbosity=-1,
)
NUM_ROUNDS = 8000
EARLY_STOP = 200
STAGE5_ANCHOR = 0.27402
# ----------------------------------------------------------------

In [3]:
# ---------------------------------------------------------------- grammar features
def build_grammar_features(fz, date_from=None, date_to=None):
    blocks = []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        n = S["n"]
        cols = []
        for c in fz.classes:
            P = S["P"][c]
            ind = np.diff(P).astype(np.float64)
            occ = np.where(ind > 0, np.arange(n), -1)
            last = np.maximum.accumulate(occ)
            lastm1 = np.concatenate(([-1], last[:-1]))
            bsince = np.where(lastm1 >= 0, np.arange(n) - lastm1, np.arange(n) + 1)
            cols.append(bsince[t])
            x = np.concatenate(([0.0], ind[:-1]))
            for tau in TAUS:
                a = np.exp(-1.0 / tau)
                s = lfilter([a], [1.0, -a], x)
                cols.append(s[t])
        lt = np.where(t > 0, S["lt_incl"][np.maximum(t - 1, 0)], -1)
        age = np.where(lt >= 0, t - lt, t + 1)
        cols.append(age)
        cols.append(S["tod"][t].astype(np.float64))
        blocks.append(np.stack(cols, 1).astype(np.float32))
    return np.concatenate(blocks), fz.grammar_names

In [4]:
# ---------------------------------------------------------------- value features
def value_base_names(manifest):
    """Value channels derived from the manifest stream set (source columns)."""
    cols = manifest["_stream_cols"]
    names = ([f"z_{c}_signed" for c in cols]
             + [f"z_{c}_mag" for c in cols]
             + ["z_leg_amp", f"z_body_{BODY_TAG}_signed", f"z_body_{BODY_TAG}_mag"])
    return cols, names


def build_value_features(fz, src, date_from=None, date_to=None):
    cols, base = value_base_names(fz.manifest)
    names = base + [f"{nm}_lag{L}" for L in VALUE_LAGS for nm in base]
    sv = src.set_index("timestamp")
    blocks = []
    for S in fz._selected(date_from, date_to):
        ts = pd.DatetimeIndex(S["timestamp"])
        r = sv.reindex(ts)
        leg_dir = S["leg_dir"]
        feats = []
        for c in cols:                                             # signed per stream col
            feats.append(_expanding_z(r[c].to_numpy(np.float64) * leg_dir, ZWARM))
        for c in cols:                                             # magnitude per stream col
            feats.append(_expanding_z(np.abs(r[c].to_numpy(np.float64)), ZWARM))
        feats.append(_expanding_z(np.abs(r["JMA"].to_numpy(np.float64) - S["leg_start_jma"]), ZWARM))
        bo = r[BODY_OPEN_COL].to_numpy(np.float64)
        bc = r[BODY_CLOSE_COL].to_numpy(np.float64)
        feats.append(_expanding_z((bc - bo) * leg_dir, ZWARM))     # confirming-positive
        feats.append(_expanding_z(np.abs(bc - bo), ZWARM))

        M = np.stack(feats, 1)
        M = np.concatenate([np.zeros((1, M.shape[1])), M[:-1]], 0)          # t-1 shift
        lagged = [M]
        for L in VALUE_LAGS:
            lagged.append(np.concatenate([np.zeros((L, M.shape[1])), M[:-L]], 0))
        M = np.concatenate(lagged, 1)
        t = np.nonzero(~S["warm"])[0]
        Mt = M[t]
        Mt[(t < ZWARM + max(VALUE_LAGS))] = 0.0
        blocks.append(Mt.astype(np.float32))
    return np.concatenate(blocks), names


def build_X(fz, src, date_from=None, date_to=None):
    Xg, gn = build_grammar_features(fz, date_from, date_to)
    Xv, vn = build_value_features(fz, src, date_from, date_to)
    assert len(Xg) == len(Xv), (len(Xg), len(Xv))
    return np.hstack([Xg, Xv]), gn + vn


def build_meta(fz, date_from=None, date_to=None):
    bi, ts, tg, dt = [], [], [], []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        bi.append(S["bar_index"][t])
        ts.append(S["timestamp"][t])
        tg.append(S["tgt"][t])
        dt.append(np.full(len(t), str(S["sess"])))
    return pd.DataFrame({"bar_index": np.concatenate(bi),
                         "timestamp": np.concatenate(ts),
                         "is_target": np.concatenate(tg),
                         "date": np.concatenate(dt)})

In [5]:
# ---------------------------------------------------------------- train / eval
def train(fz, src, train_end, valid_from):
    X, names = build_X(fz, src, None, train_end)
    meta = build_meta(fz, None, train_end)
    y = meta["is_target"].to_numpy().astype(np.int8)
    va = (meta["date"] >= valid_from).to_numpy()
    tr = ~va
    dtr = lgb.Dataset(X[tr], label=y[tr], feature_name=names)
    dva = lgb.Dataset(X[va], label=y[va], reference=dtr)
    booster = lgb.train(LGBM_PARAMS, dtr, num_boost_round=NUM_ROUNDS,
                        valid_sets=[dva], valid_names=["valid"],
                        callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False),
                                   lgb.log_evaluation(200)])
    p_va = booster.predict(X[va], num_iteration=booster.best_iteration)
    iso = IsotonicRegression(out_of_bounds="clip").fit(p_va, y[va])
    print(json.dumps(dict(n_train=int(tr.sum()), n_valid=int(va.sum()),
                          best_iteration=int(booster.best_iteration),
                          valid_logloss_cal=float(log_loss(y[va], iso.predict(p_va)))),
                     indent=2))
    imp = pd.DataFrame({"feature": names,
                        "gain": booster.feature_importance("gain")}
                       ).sort_values("gain", ascending=False)
    print(imp.head(20).to_string(index=False))
    return dict(booster=booster, iso=iso, feature_names=names,
                valid_from=valid_from, train_end=train_end,
                manifest=fz.manifest, tag=STAGE0_TAG, importance=imp)


def evaluate(fz, src, model, start, end=None, anchor=STAGE5_ANCHOR):
    X, _ = build_X(fz, src, start, end)
    meta = build_meta(fz, start, end)
    y = meta["is_target"].to_numpy().astype(np.int8)
    p = model["booster"].predict(X, num_iteration=model["booster"].best_iteration)
    p_cal = model["iso"].predict(p)
    ll_cal = log_loss(y, p_cal)
    ll_const = log_loss(y, np.full_like(p, y.mean(), dtype=np.float64))
    print(json.dumps(dict(n_rows=int(len(y)), holdout_logloss_cal=float(ll_cal),
                          holdout_logloss_const=float(ll_const),
                          skill=float(1 - ll_cal / ll_const),
                          auc=float(roc_auc_score(y, p)),
                          anchor=anchor, delta=float(ll_cal - anchor)), indent=2))
    out = meta[["bar_index", "timestamp", "is_target"]].copy()
    out["p"] = p.astype(np.float32)
    out["p_cal"] = p_cal.astype(np.float32)
    tbl = out.assign(bin=pd.qcut(out["p_cal"], 10, duplicates="drop")).groupby(
        "bin", observed=True).agg(mean_p=("p_cal", "mean"),
                                  realized=("is_target", "mean"), n=("p_cal", "size"))
    print(tbl.to_string())
    return out

In [6]:
# ---------------------------------------------------------------- run
manifest = load_manifest(MANIFEST_PATH,TOD_BIN_MIN)

SOURCE_PATH = manifest["source_file"]
RAW_FILE = manifest["raw_file"]

assert STAGE0_TAG == manifest["stage0_tag"]

bars = pd.read_parquet(BARS_PATH)
events = pd.read_parquet(EVENTS_PATH)

sess_lo = pd.Timestamp(manifest["session_start"]).time()
sess_hi = pd.Timestamp(manifest["session_end"]).time()

src = pd.read_parquet(SOURCE_PATH)
src = src[(src["timestamp"].dt.time >= sess_lo) & (src["timestamp"].dt.time < sess_hi)]

raw1 = pd.read_parquet(RAW_FILE)
raw1 = raw1[(raw1["timestamp"].dt.time >= sess_lo) & (raw1["timestamp"].dt.time < sess_hi)]
raw1 = raw1.rename(columns={"Open": "rawOpen", "High": "rawHigh", "Low": "rawLow", "Last": "rawLast"})

src = src.merge(raw1[["timestamp", "rawOpen", "rawHigh", "rawLow", "rawLast"]], on="timestamp", how="left")

###  assert day start exists in src and raw ###
src_min_time = src["timestamp"].dt.time.min()
print(f'SRC MIN TIME: {src_min_time}')
assert sess_lo == src_min_time

assert src[["rawOpen", "rawLast"]].notna().all().all(), "raw OHLC has gaps vs source timestamps"

SRC MIN TIME: 09:00:00


In [8]:
print('--------------------- !! VERIFY !! ---------------------')
print(f'FRAME: {FRAME}sec, STAGE0_TAG: {STAGE0_TAG}, BODY_TAG: {BODY_TAG}')
print('----------------------- MANIFEST -----------------------')
print(json.dumps({k: v for k, v in manifest.items() if not k.startswith("_")}, indent=2))
print('--------------------------------------------------------')

#

fz = Featurizer(bars, events, manifest, TOD_BIN_MIN, TAUS)
#augment_featurizer(fz, bars)

S0 = fz.sessions[len(fz.sessions) // 2]
xchk = src.set_index("timestamp").reindex(pd.DatetimeIndex(S0["timestamp"]))["jmaD1"].to_numpy(np.float64)
print("welford max abs diff:", _welford_check(xchk, ZWARM))

model = train(fz, src, TRAIN_END, VALID_FROM)
pred = evaluate(fz, src, model, TEST_FROM)

print(f"-------------- {STAGE0_TAG} {FRAME}s --------------")
joblib_file = f"{OUT_DIR}/model_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.joblib"
importance_file = f"{OUT_DIR}/importance_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.csv"
pred_file = f"{OUT_DIR}/pred_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.pqt"

joblib.dump({k: v for k, v in model.items() if k != "importance"}, joblib_file)
model["importance"].to_csv(importance_file, index=False)
pred.to_parquet(pred_file, index=False)

print(f'    joblib_file: {joblib_file}')
print(f'importance_file: {importance_file}')
print(f'      pred_file: {pred_file}')

#

z = pred.timestamp >= TEST_FROM
print("holdout-window logloss:", log_loss(pred.is_target[z], pred.p_cal[z]))

--------------------- !! VERIFY !! ---------------------
FRAME: 3sec, STAGE0_TAG: nq-TICK-9-12am, BODY_TAG: raw
----------------------- MANIFEST -----------------------
{
  "frame_seconds": 3,
  "session_start": "09:00",
  "session_end": "12:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "NQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "NQ_D2",
      "column": "jmaD2"
    },
    {
      "name": "TICK_D1",
      "column": "tickJmaD1"
    },
    {
      "name": "TICK_D2",
      "column": "tickJmaD2"
    }
  ],
  "self_stream": "NQ_JMA_SELF",
  "source_file": "data/nq-tick-full-3sec.pqt",
  "raw_file": "data/nq-ohlc-raw-3sec.pqt",
  "n_bars": 4106941,
  "n_sessions": 1146,
  "date_min": "2022-01-03 00:00:00",
  "date_max": "2026-07-10 00:00:00",
  "created": "2026-07-17T18:38:55.143873",
  "stage0_tag": "nq-TICK-9-12am"
}
--------------------------------------------------------
welford max abs diff: 6.217248937900877e-15
[200]	valid's binary_logloss: 0.139728
[400

In [9]:
# visual check
print(f"-------------- bars --------------")
print(bars.head())
print(f"\n-------------- events --------------")
print(events.head())
print(f"\n-------------- model --------------")
print(model)
print(f"\n-------------- pred --------------")
print(pred.head())

-------------- bars --------------
   bar_index           timestamp       date           jma        d1  \
0          0 2022-01-03 09:00:00 2022-01-03  16384.410156 -1.539062   
1          1 2022-01-03 09:00:03 2022-01-03  16382.654297 -2.816406   
2          2 2022-01-03 09:00:06 2022-01-03  16380.880859 -3.529297   
3          3 2022-01-03 09:00:09 2022-01-03  16378.742188 -3.912109   
4          4 2022-01-03 09:00:12 2022-01-03  16376.950195 -3.930664   

   jma_leg_dir  leg_id  leg_age   leg_amp  is_target  warm  
0            0       0        0  0.000000      False  True  
1           -1       0        1  1.755859      False  True  
2           -1       0        2  3.529297      False  True  
3           -1       0        3  5.667969      False  True  
4           -1       0        4  7.459961      False  True  

-------------- events --------------
        date   stream  event_bar  extremum_bar           timestamp  \
0 2022-01-03    NQ_D1         12            11 2022-01-03 09:00:

In [13]:
p = pd.read_parquet("stage-5/pred_raw_nq-TICK-9-12am_3s.pqt")
p['date'] = p['timestamp'].dt.normalize()
print(p.info())
h = p[p["date"] >= "2026-01-01"]          # no-op if the parquet is holdout-only
GREEN = float(h["p_cal"].quantile(0.50))
RED   = float(h["p_cal"].quantile(0.90))
g = h[h["p_cal"] <  GREEN]
r = h[h["p_cal"] >= RED]
print(f"GREEN {GREEN:.6g}  {len(g)/len(h):.2%} of bars  wrong 1/{1/g['p'].mean():.0f}")
print(f"RED   {RED:.6g}  {len(r)/len(h):.2%} of bars  right {r['p'].mean():.1%}")

<class 'pandas.DataFrame'>
RangeIndex: 477674 entries, 0 to 477673
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   bar_index  477674 non-null  int64         
 1   timestamp  477674 non-null  datetime64[us]
 2   is_target  477674 non-null  bool          
 3   p          477674 non-null  float32       
 4   p_cal      477674 non-null  float32       
 5   date       477674 non-null  datetime64[us]
dtypes: bool(1), datetime64[us](2), float32(2), int64(1)
memory usage: 15.0 MB
None
GREEN 0.00305229  48.90% of bars  wrong 1/1382
RED   0.385621  10.00% of bars  right 73.1%
